# `nse-backtester`: Quantitative Equity Backtesting Framework
## Production-Grade Simulation on the National Stock Exchange of India (NSE)

**Operating Context**: Quantitative Research & Strategy Evaluation  
**Universe**: Liquid Nifty 50 Constituents (Adjusted for Splits/Dividends/Bonuses)  
**Benchmark**: Nifty 50 Index (`^NSEI`)  
**Evaluation Period**: 2016 – 2025  
**Friction Calibration**: 15 bps per side (30 bps round-trip reflecting Indian STT, GST, SEBI/Exchange turnover charges, and slippage)  
**Risk-Free Rate**: 6.5% Annualized (RBI Repo Rate Baseline)

### 1. Architectural Setup & Library Imports

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tabulate import tabulate

from src.data_loader import DataLoader
from src.strategy import MomentumStrategy, MeanReversionStrategy
from src.portfolio import PortfolioManager
from src.backtester import Backtester
from src.metrics import calculate_all_metrics
from src.report import compute_monthly_returns_matrix

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

### 2. Historical Data Ingestion & Survivorship Bias Disclosure

> **Survivorship Bias Disclosure**:
> Free public data feeds such as Yahoo Finance utilize current constituent definitions. In historical backtesting, companies that were delisted, merged, or declared bankrupt during the 2016–2025 window are excluded from the dataset. In institutional deployment, point-in-time constituent archives from NSE India must be integrated to eliminate upward selection bias.

In [ ]:
loader = DataLoader(cache_dir=str(root_dir / 'data'))
prices, benchmark = loader.fetch_data(start_date='2016-01-01', end_date='2025-12-31')

print(f"Universe Asset Count: {prices.shape[1]} liquid equities")
print(f"Historical Horizon  : {len(prices)} trading days ({prices.index[0].date()} to {prices.index[-1].date()})")
print(f"Benchmark Series    : {benchmark.name} ({len(benchmark)} observations)")
prices.tail()

### 3. Quantitative Strategies Formulation

#### Strategy 1: 12-1 Month Cross-Sectional Momentum (Jegadeesh & Titman)
* **Lookback**: 252 trading days
* **Skip**: Most recent 21 trading days (eliminates short-term microstructure reversal)
* **Universe Ranking**: Cross-sectional ranking on $R_{i, t} = \frac{P_{i, t-21}}{P_{i, t-252}} - 1$
* **Portfolio Selection**: Top quintile (top 20%) equal-weighted, rebalanced monthly

#### Strategy 2: 20-Day Rolling Z-Score Mean Reversion
* **Signal**: $Z_t = \frac{P_t - \mu_{20, t}}{\sigma_{20, t}}$
* **Entry**: Long when $Z_t < -1.5$ (oversold dislocation)
* **Exit**: When $Z_t \ge 0.0$ (mean reversion completed)
* **Allocation**: Inverse-volatility weighted across active signals

In [ ]:
pm = PortfolioManager(max_weight=0.15, cash_buffer=0.05, max_gross_exposure=0.95)

# 1. Momentum Strategy
mom_strat = MomentumStrategy(lookback_days=252, skip_days=21, top_quantile=0.20, rebalance_freq='monthly')
raw_mom_weights = mom_strat.generate_signals(prices)
mom_weights = pm.process_weights(raw_mom_weights)

# 2. Mean Reversion Strategy
mr_strat = MeanReversionStrategy(window=20, entry_z=-1.5, exit_z=0.0, weighting='inverse_vol')
raw_mr_weights = mr_strat.generate_signals(prices)
mr_weights = pm.process_weights(raw_mr_weights)

print("Momentum Target Weights (last 5 days):")
display(mom_weights.tail())

### 4. Vectorised Simulation Engine (Strict Lag & Transaction Friction)

The engine enforces:
1. **Zero Look-Ahead Bias**: Execution strictly takes place on $W_{\text{exec}, t} = W_{t-1}$.
2. **Indian Market Realism**: 15 bps one-way (STT, Stamp Duty, GST, exchange charges, slippage) charged on turnover.

In [ ]:
engine = Backtester(cost_per_side=0.0015, rf_annual=0.065, initial_capital=1_000_000.0)

res_mom = engine.run(prices, mom_weights, benchmark=benchmark, strategy_name='Momentum (12-1M)')
res_mr = engine.run(prices, mr_weights, benchmark=benchmark, strategy_name='Mean Reversion (Z-Score)')

print("Backtest Simulation Successful!")

### 5. Quantitative Performance Metrics Comparison

In [ ]:
from main import format_summary_table
print(format_summary_table([res_mom, res_mr]))

### 6. Interactive Visualizations

#### 6.1 Cumulative Equity Progression & Underwater Drawdowns

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True, gridspec_kw={'height_ratios': [2.2, 1.0]})

hist_mom = res_mom.portfolio_history
hist_mr = res_mr.portfolio_history
dates = hist_mom.index

# Equity Curves
ax1.plot(dates, hist_mom['equity_curve'] / 1e6, label=f'Momentum (12-1M) [CAGR: {res_mom.metrics["CAGR"]*100:.1f}%]', color='#1f77b4', lw=2)
ax1.plot(dates, hist_mr['equity_curve'] / 1e6, label=f'Mean Reversion [CAGR: {res_mr.metrics["CAGR"]*100:.1f}%]', color='#2ca02c', lw=1.8)
ax1.plot(dates, hist_mom['benchmark_equity_curve'] / 1e6, label=f'Nifty 50 (^NSEI) [CAGR: {res_mom.benchmark_metrics["CAGR"]*100:.1f}%]', color='#7f7f7f', lw=1.6, ls='--')

ax1.set_title('Cumulative Equity Progression: Momentum vs Mean Reversion vs Nifty 50 (2016-2025)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Portfolio Value (Million INR)', fontsize=11, fontweight='bold')
ax1.legend(loc='upper left', frameon=True, fontsize=10)
ax1.grid(True, linestyle=':', alpha=0.6)

# Underwater Drawdowns
ax2.plot(dates, hist_mom['drawdown'] * 100, label='Momentum Drawdown', color='#1f77b4', lw=1.2)
ax2.fill_between(dates, hist_mom['drawdown'] * 100, 0, color='#1f77b4', alpha=0.15)
ax2.plot(dates, hist_mom['benchmark_drawdown'] * 100, label='Nifty 50 Drawdown', color='#7f7f7f', lw=1.2, ls=':')

ax2.set_title('Underwater Drawdown Comparison (%)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Drawdown (%)', fontsize=11, fontweight='bold')
ax2.set_xlabel('Date', fontsize=11, fontweight='bold')
ax2.legend(loc='lower left', frameon=True, fontsize=9)
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

#### 6.2 Momentum Monthly Returns Heatmap

In [ ]:
monthly_m = compute_monthly_returns_matrix(res_mom.portfolio_history['net_return'])

fig, ax = plt.subplots(figsize=(13, 6))
matrix_vals = monthly_m.to_numpy() * 100.0

cax = ax.matshow(matrix_vals, cmap='RdYlGn', vmin=-12, vmax=12)
fig.colorbar(cax, ax=ax, fraction=0.03, pad=0.04, label='Return (%)')

ax.set_xticks(range(len(monthly_m.columns)))
ax.set_xticklabels(monthly_m.columns, fontsize=10, fontweight='bold')
ax.set_yticks(range(len(monthly_m.index)))
ax.set_yticklabels(monthly_m.index, fontsize=10, fontweight='bold')

for i in range(len(monthly_m.index)):
    for j in range(len(monthly_m.columns)):
        val = matrix_vals[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:+.1f}%", ha='center', va='center', color='black', fontsize=8)

ax.set_title('Momentum Strategy Monthly Returns Matrix (2016-2025)', fontsize=13, fontweight='bold', pad=15)
plt.grid(False)
plt.show()